In [1]:
from singleCAM_IROS._pipeline_support import _handle_dirpaths

from pathlib import Path

from astropy.io import fits
from astropy.io.fits.fitsrec import FITS_rec
from numpy.typing import NDArray
import numpy as np
from scipy.interpolate import griddata

from bloodmoon.mask import CodedMaskCamera, codedmask

import darksun as ds

ds.show.set_figures_darkbkg()

In [2]:
def load_fits_data(path: Path, ext: int = 1) -> FITS_rec:
    """Loads the FITS file data from chosen extension."""
    return fits.getdata(path, ext=ext, header=False)

def extract_spectrum(
    data: FITS_rec,
    source_idx: int,
    energy_range: tuple[float, float] = (2.0, 50.0),
) -> tuple[NDArray, NDArray]:
    """
    Extracts source energy and spectrum values in specified `energy_range`.
    """
    energy: NDArray = data['ENERGY'][source_idx][:-1]
    low, high = energy_range
    band: NDArray = (energy >= low) & (energy <= high)
    spectrum: NDArray = data['SPECTRUM'][source_idx]
    return energy[band], spectrum[band]

def extract_transmission(
    data: FITS_rec,
    energy_range: tuple[float, float] = (2.0, 50.0),
) -> tuple[NDArray, NDArray]:
    """
    Extracts photons transmission values in specified `energy_range`.
    """
    energy: NDArray = data.field(0)
    transmission: NDArray = data.field(1)
    low, high = energy_range
    band: NDArray = (energy >= low) & (energy <= high)
    return energy[band], transmission[band]

In [3]:
def interp(
    x: NDArray,
    y: NDArray,
    energy: NDArray,
) -> NDArray:
    """Interpolates `y` values in given `energy` values."""
    # ...
    # some preprocess (?)
    # ...
    return griddata(x, y, energy, method='linear')

def integrate(arr: NDArray, bins: NDArray) -> float:
    """Integrates input array."""
    int_: NDArray = np.cumsum(arr * bins)
    return int_[-1]

In [ ]:
def _local2polar(theta: NDArray) -> float:
    """
    Computes transformation between local-frame
    and polar frame. Input `theta` are in [rad].
    """
    return np.sqrt(np.sum(np.square(np.tan(theta))))

def compute_theta(theta_x: float, theta_y: float) -> float:
    """
    Computes the polar angular coord wrt to the xy plane.
    Both `theta_x` and `theta_y` are in [deg].
    Output angle value is in [deg].
    """
    local_angles: NDArray = np.deg2rad(np.array([theta_x, theta_y]))
    xi: float = _local2polar(local_angles)
    return np.rad2deg(np.atan(xi))

def project_absrp_correction(
    distance: float,
    theta_x: float,
    theta_y: float,
) -> NDArray:
    """
    Computes the source local-frame coords correction for the
    detector absorption photons distance in a given energy band.
    """
    local_angles: NDArray = np.deg2rad(np.array([theta_x, theta_y]))
    xi: float = _local2polar(local_angles)
    theta_plane_proj: float = np.sin(np.atan(xi))
    phi_local_proj: NDArray = np.tan(local_angles) / (xi + 1e-8)
    return distance * theta_plane_proj * phi_local_proj

In [5]:
#settings_path: str = "/mnt/dbb8f47e-da06-47bf-8ef5-038092af70f7/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Data/Simulations/camera_settings"
settings_path: str = "/mnt/d/PhD_AASS/Coding/Images_fits/camera_settings"

sources_path: Path = Path(settings_path, "RXTE-ASM_BeppoSAX-WFC_catalog_new_2-50keV.fits")

detSi_matten_path: Path = Path(settings_path, "detectorSi_absrp.fits")
detBe_dl_filter_path: Path = Path(settings_path, "detectorBe_deadlayer_filter.fits")

maskKapton_mli_path: Path = Path(settings_path, "mask_MLI_Kapton.fits")

sources: FITS_rec = load_fits_data(sources_path)
detSi_matten: FITS_rec = load_fits_data(detSi_matten_path)
detBe_dl_filter: FITS_rec = load_fits_data(detBe_dl_filter_path)
maskKapton_mli: FITS_rec = load_fits_data(maskKapton_mli_path)

**Source Energy Band Counts**

In [20]:
from darksun.benchmarking import source_catalogue_data

def _setup_integral_arrs(
    x: NDArray,
    y: NDArray,
    interval: tuple[float, float],
) -> tuple[NDArray, NDArray]:
    """
    Extract `x` values in the `interval` range and respective
    values for `y`. Since x represents a binning array, len(x)
    must be equal to len(y) + 1.
    """
    low, high = interval
    band: NDArray = np.where((x >= low) & (x <= high))[0]
    return x[band], y[band[:-1]]

def extract_spectrum(
    data: FITS_rec,
    source_idx: int,
    energy_range: tuple[float, float] = (2.0, 50.0),
) -> tuple[NDArray, NDArray]:
    """
    Extracts source energy and spectrum values in specified `energy_range`.
    """
    energy: NDArray = data['ENERGY'][source_idx][:-1]
    low, high = energy_range
    band: NDArray = (energy >= low) & (energy <= high)
    spectrum: NDArray = data['SPECTRUM'][source_idx]
    return energy[band], spectrum[band]

def compute_source_avgflux(
    data: FITS_rec,
    source_idx: int,
    energy_range: tuple[float, float] = (2.0, 50.0),
) -> float:
    """Computes a source average flux [ph/cm2/s] in given energy range."""
    # - energy values are considered as bins, so we have:
    #   len(energy) = len(spectrum) + 1
    # - to use the energy bins in the given energy range,
    #   we need to remove the last element on the spectr arr
    energy, spectrum = _setup_integral_arrs(
        data[source_idx]['ENERGY'],
        data[source_idx]['SPECTRUM'],
        energy_range,
    )
    return integrate(spectrum, np.diff(energy))

In [19]:
source_idx: int = 0
energy_band: tuple[float, float] = (2.0, 6.0)   # [keV]


compute_source_avgflux(sources, source_idx, energy_band)

np.float32(0.0178953)

In [23]:
len(source_catalogue_data('crab', sources)['ENERGY'])

513